In [1]:
import cv2
import numpy as np
import os
import glob
from IPython.display import clear_output

# Define your input and output folders
input_folder = './public/frames'
output_folder = './public/frames_enhanced'

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Grab all your ezgif frames
frame_paths = sorted(glob.glob(f"{input_folder}/*.jpg"))
total_frames = len(frame_paths)

# Sharpening kernel for crisp details
sharpen_kernel = np.array([[-1,-1,-1], 
                           [-1, 9,-1], 
                           [-1,-1,-1]])

print(f"Found {total_frames} frames. Starting enhancement pipeline...")

for index, path in enumerate(frame_paths):
    # Read the image
    img = cv2.imread(path)
    
    if img is None:
        print(f"Warning: Could not read {path}")
        continue
        
    # Step 1: Remove compression banding and artifacts
    denoised = cv2.fastNlMeansDenoisingColored(img, None, 3, 3, 7, 21)
    
    # Step 2: Cinematic Contrast & Saturation Boost
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    cl = clahe.apply(l)
    
    limg = cv2.merge((cl, a, b))
    enhanced_contrast = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    
    # Step 3: Crisp Sharpening
    final_frame = cv2.filter2D(enhanced_contrast, -1, sharpen_kernel)
    
    # Save the output
    filename = os.path.basename(path)
    cv2.imwrite(os.path.join(output_folder, filename), final_frame)
    
    # Display progress updates
    if (index + 1) % 10 == 0 or (index + 1) == total_frames:
        clear_output(wait=True)
        print(f"Processed {index + 1}/{total_frames} frames ({(index+1)/total_frames*100:.1f}%)")

print("Enhancement complete! High-quality frames are in /public/frames_enhanced")

Processed 100/300 frames (33.3%)


KeyboardInterrupt: 

In [2]:
import cv2
import numpy as np
import os
import glob
import shutil
from IPython.display import clear_output

input_folder = './public/frames'
output_folder = './public/frames_enhanced'

# Automatically delete the old folder and its contents if they exist
if os.path.exists(output_folder):
    print("Clearing out old degraded frames...")
    shutil.rmtree(output_folder)

# Create a fresh, empty output directory
os.makedirs(output_folder, exist_ok=True)

frame_paths = sorted(glob.glob(f"{input_folder}/*.jpg"))
total_frames = len(frame_paths)

print(f"Found {total_frames} frames. Starting gentle upscaling...")

for index, path in enumerate(frame_paths):
    img = cv2.imread(path)
    if img is None: continue
    
    # 1. Gentle Denoising (smooths out the Ezgif blockiness)
    denoised = cv2.fastNlMeansDenoisingColored(img, None, 4, 4, 7, 21)
    
    # 2. Smooth Upscaling (2x resolution using Cubic interpolation)
    height, width = denoised.shape[:2]
    upscaled = cv2.resize(denoised, (width * 2, height * 2), interpolation=cv2.INTER_CUBIC)
    
    # 3. Mild Saturation Boost (makes the lighting/greens pop without blowing out contrast)
    hsv = cv2.cvtColor(upscaled, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    # Boost saturation by 15%
    s = cv2.multiply(s, 1.15) 
    s = np.clip(s, 0, 255).astype(hsv.dtype)
    
    hsv = cv2.merge((h, s, v))
    final_frame = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    
    # Save output
    filename = os.path.basename(path)
    cv2.imwrite(os.path.join(output_folder, filename), final_frame)
    
    if (index + 1) % 10 == 0 or (index + 1) == total_frames:
        clear_output(wait=True)
        print(f"Processed {index + 1}/{total_frames} frames ({(index+1)/total_frames*100:.1f}%)")

Processed 240/300 frames (80.0%)


KeyboardInterrupt: 

In [5]:
%pip install Pillow

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.2 MB 9.6 MB/s eta 0:00:01
   ----------------- ---------------------- 3.1/7.2 MB 10.5 MB/s eta 0:00:01
   ---------------------------------- ----- 6.3/7.2 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------  7.1/7.2 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 9.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from PIL import Image

# Point to your existing heavy frames and create a new folder for the lightweight ones
input_folder = 'public/frames_raw'
output_folder = 'public/frames_optimized'

# Create the optimized folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Get all PNG files and sort them to keep the frame order
files = sorted([f for f in os.listdir(input_folder) if f.endswith('.png')])
total_files = len(files)

print(f"Starting conversion of {total_files} frames to WebP...")

for i, filename in enumerate(files, 1):
    input_path = os.path.join(input_folder, filename)
    
    # Change the extension from .png to .webp
    new_filename = filename.replace('.png', '.webp')
    output_path = os.path.join(output_folder, new_filename)
    
    # Open and convert the image
    with Image.open(input_path) as img:
        # quality=75 is the industry sweet spot for balancing sharp visuals and tiny file sizes
        # method=6 tells the encoder to use its maximum compression effort
        img.save(output_path, 'webp', quality=75, method=6)
        
    # Print progress every 20 frames so you know it's working
    if i % 20 == 0 or i == total_files:
        print(f"Converted {i}/{total_files} frames...")

print("✅ All frames successfully optimized to WebP!")

Starting conversion of 240 frames to WebP...
Converted 20/240 frames...
Converted 40/240 frames...
Converted 60/240 frames...
Converted 80/240 frames...
Converted 100/240 frames...
Converted 120/240 frames...
Converted 140/240 frames...
Converted 160/240 frames...
Converted 180/240 frames...
Converted 200/240 frames...
Converted 220/240 frames...
Converted 240/240 frames...
✅ All frames successfully optimized to WebP!


In [7]:
import cv2
import os
import shutil
from PIL import Image

# 1. Define paths
video_path = 'video.mp4'  # IMPORTANT: Change this if your video file has a different name
raw_folder = 'public/frames_raw'
optimized_folder = 'public/frames_optimized'
target_frame_count = 240

# 2. Wipe the slate clean (Delete unwanted/old folders)
if os.path.exists(raw_folder):
    shutil.rmtree(raw_folder)
    print(f"🗑️ Deleted old {raw_folder} folder.")

if os.path.exists(optimized_folder):
    shutil.rmtree(optimized_folder)
    print(f"🗑️ Deleted old {optimized_folder} folder.")

# 3. Create fresh output folder
os.makedirs(optimized_folder, exist_ok=True)

# 4. Extract and optimize frames directly
cap = cv2.VideoCapture(video_path)
frame_count = 0

print("🚀 Starting direct extraction and high-quality WebP optimization...")

while True:
    success, frame = cap.read()
    if not success or frame_count >= target_frame_count:
        break
        
    frame_count += 1
    
    # Convert OpenCV's BGR color format to standard RGB for Pillow
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb_frame)
    
    # Save directly as high-quality WebP
    filename = f"frame_{str(frame_count).zfill(3)}.webp"
    output_path = os.path.join(optimized_folder, filename)
    
    # quality=98 for near-lossless clarity, method=6 for maximum compression efficiency
    img.save(output_path, 'webp', quality=98, method=6)
    
    # Print progress
    if frame_count % 20 == 0 or frame_count == target_frame_count:
        print(f"✅ Processed {frame_count}/{target_frame_count} frames...")

cap.release()
print("🎉 All frames successfully extracted and saved directly to WebP!")

🗑️ Deleted old public/frames_optimized folder.
🚀 Starting direct extraction and high-quality WebP optimization...
🎉 All frames successfully extracted and saved directly to WebP!


In [8]:
import cv2
import os
import shutil
from PIL import Image

# 1. Define paths using your exact video location
video_path = r'C:\Users\Rajab\Desktop\Work\gemini_generated_video_74e4bb56.mp4'
raw_folder = 'public/frames_raw'
optimized_folder = 'public/frames_optimized'
target_frame_count = 240

# 2. Wipe the slate clean (Delete unwanted/old folders)
if os.path.exists(raw_folder):
    shutil.rmtree(raw_folder)
    print(f"🗑️ Deleted old {raw_folder} folder.")

if os.path.exists(optimized_folder):
    shutil.rmtree(optimized_folder)
    print(f"🗑️ Deleted old {optimized_folder} folder.")

# 3. Create fresh output folder
os.makedirs(optimized_folder, exist_ok=True)

# 4. Extract and optimize frames directly
cap = cv2.VideoCapture(video_path)
frame_count = 0

print("🚀 Starting direct extraction and high-quality WebP optimization...")

while True:
    success, frame = cap.read()
    if not success or frame_count >= target_frame_count:
        break
        
    frame_count += 1
    
    # Convert OpenCV's BGR color format to standard RGB for Pillow
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb_frame)
    
    # Save directly as high-quality WebP
    filename = f"frame_{str(frame_count).zfill(3)}.webp"
    output_path = os.path.join(optimized_folder, filename)
    
    # quality=98 for near-lossless clarity, method=6 for maximum compression efficiency
    img.save(output_path, 'webp', quality=98, method=6)
    
    # Print progress
    if frame_count % 20 == 0 or frame_count == target_frame_count:
        print(f"✅ Processed {frame_count}/{target_frame_count} frames...")

cap.release()
print("🎉 All frames successfully extracted and saved directly to WebP!")

🗑️ Deleted old public/frames_optimized folder.
🚀 Starting direct extraction and high-quality WebP optimization...
✅ Processed 20/240 frames...
✅ Processed 40/240 frames...
✅ Processed 60/240 frames...
✅ Processed 80/240 frames...
✅ Processed 100/240 frames...
✅ Processed 120/240 frames...
✅ Processed 140/240 frames...
✅ Processed 160/240 frames...
✅ Processed 180/240 frames...
✅ Processed 200/240 frames...
✅ Processed 220/240 frames...
✅ Processed 240/240 frames...
🎉 All frames successfully extracted and saved directly to WebP!


In [1]:
import cv2
import os

video_path = 'public/optimized_scrub.mp4'
output_folder = 'public/frames'

# Create the frames folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

vidcap = cv2.VideoCapture(video_path)
success, image = vidcap.read()
count = 1

# Extract frames as PNGs (so they are ready for Upscayl)
while success:
    cv2.imwrite(os.path.join(output_folder, f"frame_{count:03d}.png"), image)
    success, image = vidcap.read()
    count += 1

print(f"Extraction complete! Saved {count-1} frames.")

Extraction complete! Saved 240 frames.


In [1]:
import cv2
import os

# The folder where Upscayl saved the massive PNGs
input_folder = 'public/frames/upscayl_png_upscayl-lite-4x_4x'

# The folder where the React component will look for the WebP files
output_folder = 'public/frames'

print("Starting 4K WebP conversion...")

# Loop through all 240 frames
for i in range(1, 241):
    png_filename = f"frame_{i:03d}.png"
    webp_filename = f"frame_{i:03d}.webp"
    
    input_path = os.path.join(input_folder, png_filename)
    output_path = os.path.join(output_folder, webp_filename)
    
    if os.path.exists(input_path):
        img = cv2.imread(input_path)
        # Convert to WebP at 90% quality for a perfect balance of 4K crispness and tiny file size
        cv2.imwrite(output_path, img, [cv2.IMWRITE_WEBP_QUALITY, 90])
    else:
        print(f"Could not find {png_filename}")

print("WebP conversion complete! Your frames are ready.")

Starting 4K WebP conversion...
WebP conversion complete! Your frames are ready.
